# Benchmark from Label Arrays

Evaluate predictions when you already have integer label arrays (e.g., from model inference). No GFF files needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dna_segmentation_benchmark import (
    LabelConfig, EvalMetrics, benchmark_gt_vs_pred_multiple, compare_multiple_predictions,
)

label_config = LabelConfig(
    labels={0: "EXON", 2: "INTRON", 8: "NONCODING"},
    background_label=8,
    coding_label=0,
)

## Load your arrays

Replace this cell with your own data loading. Each element in the list is one sequence (e.g., one transcript). GT and pred arrays must be the same length per sequence.

In [ ]:
# --- Synthetic example data (replace with your own arrays) ---
rng = np.random.default_rng(42)
n_sequences = 200

def make_gene_sequence(length: int, rng: np.random.Generator) -> np.ndarray:
    """Generate a synthetic gene annotation array with exon/intron/noncoding."""
    arr = np.full(length, 8, dtype=np.int32)  # background
    pos = rng.integers(50, 200)
    while pos < length - 100:
        exon_len = rng.integers(50, 300)
        arr[pos:pos + exon_len] = 0  # exon
        pos += exon_len
        intron_len = rng.integers(200, 2000)
        arr[pos:pos + intron_len] = 2  # intron
        pos += intron_len
    return arr

def perturb(gt: np.ndarray, rng: np.random.Generator, noise: float = 0.05) -> np.ndarray:
    """Create a noisy prediction by randomly flipping labels."""
    pred = gt.copy()
    flip_mask = rng.random(len(gt)) < noise
    pred[flip_mask] = rng.choice([0, 2, 8], size=flip_mask.sum())
    # Shift some boundaries
    for _ in range(rng.integers(0, 5)):
        shift_pos = rng.integers(0, len(gt))
        shift_len = rng.integers(-20, 20)
        s, e = max(0, shift_pos), min(len(gt), shift_pos + abs(shift_len))
        pred[s:e] = gt[max(0, s + shift_len):max(0, s + shift_len) + (e - s)]
    return pred

gt_labels = [make_gene_sequence(rng.integers(5000, 50000), rng) for _ in range(n_sequences)]
model_a_preds = [perturb(gt, rng, noise=0.03) for gt in gt_labels]
model_b_preds = [perturb(gt, rng, noise=0.08) for gt in gt_labels]

## Benchmark and plot

In [ ]:
classes = [0, 2]  # EXON and INTRON
metrics = [
    EvalMetrics.INDEL,
    EvalMetrics.REGION_DISCOVERY,
    EvalMetrics.BOUNDARY_EXACTNESS,
    EvalMetrics.NUCLEOTIDE_CLASSIFICATION,
]

all_results = {}
for name, preds in [("model_a", model_a_preds), ("model_b", model_b_preds)]:
    all_results[name] = benchmark_gt_vs_pred_multiple(
        gt_labels=gt_labels,
        pred_labels=preds,
        label_config=label_config,
        classes=classes,
        metrics=metrics,
    )

In [ ]:
figures = compare_multiple_predictions(
    per_method_benchmark_res=all_results,
    label_config=label_config,
    classes=classes,
    metrics_to_eval=metrics,
)

for fig in figures.values():
    plt.show()